# HSM Visualiser — upload training packages

For each `data/sdm_models/<model_id>/` folder with `package.json` + `model.pkl`, finds the matching catalog row (`species` × `activity`) and **PUT**s `metadata` + `serialized_model_file`.

Credentials live in **`notebooks/.env.hsm`** (copy from `.env.hsm.example`). That file is **gitignored**.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pyhere import here

os.chdir(here())
load_dotenv(here() / "notebooks" / ".env.hsm")

In [ ]:
import json
from typing import Any, Dict, Iterator, List, Tuple

import requests

from sdm.utils.hsm_metadata import model_metadata_from_package

BASE_URL = os.environ["HSM_BASE_URL"].rstrip("/")
EMAIL = os.environ["HSM_EMAIL"]
PASSWORD = os.environ["HSM_PASSWORD"]
MODELS_DIR = Path(here() / "data" / "sdm_models")

MODELS_DIR

In [ ]:
def training_packages(models_dir: Path) -> Iterator[Tuple[Path, Path, Dict[str, Any]]]:
    for pkg_json in sorted(models_dir.glob("*/package.json")):
        pkl = pkg_json.parent / "model.pkl"
        if not pkl.is_file():
            continue
        pkg = json.loads(pkg_json.read_text(encoding="utf-8"))
        yield pkg_json, pkl, pkg


def catalog_index(models: List[Dict[str, Any]]) -> Dict[Tuple[str, str], str]:
    return {(row["species"], row["activity"]): row["id"] for row in models}


def authenticated_session(base_url: str, email: str, password: str) -> requests.Session:
    s = requests.Session()
    r = s.post(
        f"{base_url}/auth/token",
        json={"email": email, "password": password, "admin_only": True},
        timeout=60,
    )
    r.raise_for_status()
    s.headers["Authorization"] = f"Bearer {r.json()['id_token']}"
    return s

In [ ]:
session = authenticated_session(BASE_URL, EMAIL, PASSWORD)
r = session.get(f"{BASE_URL}/models", timeout=60)
r.raise_for_status()
catalog = catalog_index(r.json())
len(catalog), list(catalog.keys())[:3]

In [ ]:
missing: list[tuple[str, str, str]] = []
failed: list[tuple[str, str]] = []
ok = 0

for _pkg_json, pkl, pkg in training_packages(MODELS_DIR):
    latin = pkg.get("latin_name") or ""
    activity = pkg.get("activity_type") or ""
    slug = pkg.get("model_id") or pkl.parent.name
    model_id = catalog.get((latin, activity))
    if not model_id:
        missing.append((slug, latin, activity))
        continue

    meta = model_metadata_from_package(pkg)
    pr = session.put(
        f"{BASE_URL}/models/{model_id}",
        data={"metadata": json.dumps(meta)},
        files={
            "serialized_model_file": (
                "model.pkl",
                pkl.read_bytes(),
                "application/octet-stream",
            ),
        },
        timeout=300,
    )
    if pr.status_code != 200:
        failed.append((slug, f"HTTP {pr.status_code} {pr.text[:500]}"))
        continue
    print(f"OK {latin} — {activity} -> {model_id}")
    ok += 1

for slug, latin, activity in missing:
    print(f"MISSING API ROW {slug} ({latin!r}, {activity!r})")
for slug, err in failed:
    print(f"FAILED {slug}: {err}")

print(f"Done: {ok} uploaded, {len(missing)} no API match, {len(failed)} errors")